# Ordinary kriging: explore prediction and model variance

This lab uses the repository's tested ordinary-kriging implementation. Change the partial sill and range scale of a zero-nugget isotropic exponential variogram and inspect how the prediction and model-based variance respond.

The parameters are supplied by you; this notebook does not fit a variogram. It is a sensitivity experiment, not a model-selection procedure or a complete geostatistical study.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

current = Path.cwd().resolve()
repo_root = next((p for p in (current, *current.parents)
                  if (p / 'implementations/python/geostatistics').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
sys.path.insert(0, str(repo_root / 'implementations/python/geostatistics'))
from interpolation import ordinary_kriging_predict

rng = np.random.default_rng(42)
coordinates = rng.uniform(0, 10, size=(24, 2))
x, y = coordinates[:, 0], coordinates[:, 1]
values = np.sin(x / 2) + np.cos(y / 3) + rng.normal(0, 0.15, size=len(x))
samples = [(float(px), float(py), float(value)) for (px, py), value in zip(coordinates, values)]

grid_x = np.linspace(0, 10, 19)
grid_y = np.linspace(0, 10, 19)
xx, yy = np.meshgrid(grid_x, grid_y)
targets = list(zip(xx.ravel().tolist(), yy.ravel().tolist()))
print('Repository:', repo_root)

## 1. Inspect the sample locations

The observations follow a smooth synthetic spatial pattern with added noise. This is a teaching dataset, not a measurement campaign. The sample coordinates are unique and measured in planar coordinate units.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x, y, c=values, s=85, edgecolor='black')
fig.colorbar(sc, ax=ax, label='Synthetic observed value')
ax.set(xlabel='x coordinate', ylabel='y coordinate', title='Synthetic sample locations')
ax.set_aspect('equal')
ax.grid(alpha=0.2)
plt.show()

## 2. Change the exponential variogram parameters

The model is \(\gamma(h)=c[1-\exp(-h/a)]\), with partial sill \(c\) and range scale \(a\). The example uses a zero nugget and isotropy. The ordinary-kriging system also enforces weights that sum to one.

The variance shown is conditional on the supplied model and sample geometry. It is not an empirical confidence interval and does not include uncertainty in the variogram parameters.

In [ ]:
def explore(partial_sill=1.0, range_scale=3.0):
    results = ordinary_kriging_predict(
        samples, targets, partial_sill=partial_sill, range_scale=range_scale
    )
    prediction = np.asarray([row[0] for row in results]).reshape(xx.shape)
    variance = np.asarray([row[1] for row in results]).reshape(xx.shape)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    image0 = axes[0].imshow(prediction, origin='lower', extent=(0, 10, 0, 10), aspect='equal', cmap='viridis')
    axes[0].scatter(x, y, c='white', edgecolor='black', s=25)
    axes[0].set(title='Ordinary-kriging prediction', xlabel='x coordinate', ylabel='y coordinate')
    fig.colorbar(image0, ax=axes[0], label='Predicted value')

    image1 = axes[1].imshow(variance, origin='lower', extent=(0, 10, 0, 10), aspect='equal', cmap='magma')
    axes[1].scatter(x, y, c='white', edgecolor='black', s=25)
    axes[1].set(title='Model-based kriging variance', xlabel='x coordinate', ylabel='y coordinate')
    fig.colorbar(image1, ax=axes[1], label='Kriging variance')
    plt.show()

    print(f'Partial sill: {partial_sill:.2f}; range scale: {range_scale:.2f}')
    print('The variance is conditional on the zero-nugget isotropic exponential model.')

interact(
    explore,
    partial_sill=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description='Partial sill'),
    range_scale=FloatSlider(value=3.0, min=0.5, max=8.0, step=0.25, description='Range scale'),
);

## 3. What to investigate

1. Increase the range scale. How does the influence of distant observations change?
2. Increase the partial sill. Does the variance surface change by the same proportion as the prediction surface?
3. Where is the variance highest relative to the sample locations?
4. Why can the prediction remain smooth while the model-based variance changes?
5. Why should you validate predictions with spatially separated folds rather than relying on this variance surface alone?

## 4. Important limits

This baseline uses a dense system, a zero-nugget isotropic exponential variogram and supplied parameters. It does not fit the variogram, model a trend, support directional anisotropy or scale to large datasets. The variance is conditional on the assumed model and is not a substitute for spatial validation.

Continue with the [interpolation lesson](../algorithms/geostatistics/interpolation.md), [empirical semivariogram method](../methods/empirical-semivariogram.md), [spatial cross-validation method](../methods/spatial-cross-validation.md) and [tested implementation](../implementations/python/geostatistics/README.md).